In [2]:
# NOTE:
# This import assumes that the project root directory is in PYTHONPATH.
# It works in PyCharm's Jupyter environment by default.
# If running in standalone Jupyter Lab, you may need to adjust
# the working directory or manually append the project root to sys.path.
from data.dataProcess import *

In [3]:
# Load the normalized spatial single-cell data
adata = ad.read_h5ad('nor_log1p_codex.h5ad')

In [4]:
obs_df = adata.obs
unique_labels = obs_df['cluster_cellcharter'].unique()
num_labels = len(unique_labels)
new_labels = [f'niche{i + 1}' for i in range(num_labels)]
label_mapping = dict(zip(unique_labels, new_labels))
obs_df['cluster_cellcharter'] = obs_df['cluster_cellcharter'].map(label_mapping)
adata.obs = obs_df

In [5]:
BALBc1 = adata[adata.obs['sample'] == 'BALBc-1'].copy()
BALBc2 = adata[adata.obs['sample'] == 'BALBc-2'].copy()

In [6]:
# Remove rare niches with very few cells.
cell_type_col = "cluster_cellcharter"
BALBc1, info = filter_rare_cell_types_logspace(
    BALBc1,
    cell_type_col=cell_type_col,
    sigma_cut=0.1,
    eps=1e-8,
    verbose=True
)

log-space stats: mean=8.721, std=0.746
threshold (count): 5689
cell types to remove (3): ['niche8', 'niche7', 'niche11']
removing 7.64% of samples (6290/82382)


In [7]:
type_list = list(BALBc1.obs[cell_type_col].unique())
type_list

['niche1',
 'niche2',
 'niche3',
 'niche4',
 'niche5',
 'niche6',
 'niche9',
 'niche10']

In [8]:
# Based on the size of the blank tissue section,
# chose the sliding-window width and overlap ratio, 
# and computed the corresponding summary statistics.
coords = BALBc1.obsm["spatial"]
x = coords[:, 0]
y = coords[:, 1]

x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()
width = x_max - x_min
res = spatial_sliding_window_stats(
    BALBc1,
    window_width=width / 300,
    overlap_rate=0.8
)

print("Total number of windows:", res["n_windows"])
print("Average cells per window:", res["mean_cells_per_window"])

Total number of windows: 1495
Average cells per window: 253.97257525083612


In [9]:
dp = data_process(type_list, 'CODEX', rand_n=3000, rand_cell_num=np.floor(res["mean_cells_per_window"]),
                  label_key=cell_type_col, spatial_key='spatial')

In [10]:
train_datas = [BALBc1]
test_datas = [BALBc2]

In [11]:
train_x_sim_list = []
train_y_list = []

test_x_sim_list = []
test_y_list = []

In [12]:
angels = [0, 30, 45, 60, 90]

In [13]:
for trainData in train_datas:
    for angel in angels:
        x_sim, y = dp.generate_pseudo_bulk(trainData, angle_deg=angel,
                                           strip_width=width / 300, overlap_ratio=0.8, min_cells=res["mean_cells_per_window"]/3, min_cell_types=2)
        train_x_sim_list += x_sim
        train_y_list += y

Success rate: 100.0%
Success rate: 83.9%
Success rate: 82.3%
Success rate: 80.9%
Success rate: 100.0%


In [14]:
train_x_sim_list_, train_y_list_ = dp.build_pseudo_bulk_no_noise(BALBc1)
train_x_sim_list += train_x_sim_list_
train_y_list += train_y_list_

100%|██████████| 3000/3000 [00:22<00:00, 135.98it/s]


In [15]:
coords = BALBc2.obsm["spatial"]
x = coords[:, 0]
y = coords[:, 1]

x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()
width = x_max - x_min
res = spatial_sliding_window_stats(
    BALBc2,
    window_width=width / 300,
    overlap_rate=0.8
)

print("Total number of windows:", res["n_windows"])
print("Average cells per window:", res["mean_cells_per_window"])

Total number of windows: 1495
Average cells per window: 273.62809364548497


In [16]:
for testData in test_datas:
    for angel in angels:
        x_sim, y = dp.generate_pseudo_bulk(testData, angle_deg=angel,
                                           strip_width=width / 300, overlap_ratio=0, min_cells=res["mean_cells_per_window"]/3, min_cell_types=2)
        test_x_sim_list += x_sim
        test_y_list += y

Success rate: 100.0%
Success rate: 82.4%
Success rate: 84.2%
Success rate: 87.0%
Success rate: 100.0%


In [17]:
len(train_x_sim_list),len(test_y_list)

(10585, 1487)

In [18]:
train = [train_x_sim_list, train_y_list]
test = [test_x_sim_list, test_y_list]
with open(f'{dp.tissue_name}_nonorm', 'wb') as f:
    pickle.dump(train, f)
    pickle.dump(test, f)

In [19]:
train_x_sim_list = dp.normalize(train_x_sim_list)
test_x_sim_list = dp.normalize(test_x_sim_list)

In [20]:
train = [train_x_sim_list, train_y_list]
test = [test_x_sim_list, test_y_list]
with open(f'{dp.tissue_name}_norm', 'wb') as f:
    pickle.dump(train, f)
    pickle.dump(test, f)

In [21]:
with open(f'{dp.tissue_name}_ref', 'wb') as f:
    pickle.dump(train_datas, f)